In [1]:
import pandas as pd
import numpy as np
import openpyxl 
import os

# --- تنظیمات آدرس‌ها ---
file_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_generator_bearings_univariate\univariate\dsas_g11_generator_bearings_univariate_output2.xlsx'


# سنسورهایی که تست ۳-سیگما روی آن‌ها اجرا می‌شود
target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']

# پارامتر لاندا (ضریب وزنی)
LAMBDA = 0.2

def calculate_ewma(data, lambda_val):
    """محاسبه مقادیر EWMA به صورت بازگشتی"""
    ewma_values = np.zeros(len(data))
    if len(data) == 0:
        return ewma_values
    
    ewma_values[0] = data[0]
    for t in range(1, len(data)):
        ewma_values[t] = lambda_val * data[t] + (1 - lambda_val) * ewma_values[t-1]
    
    return ewma_values

def calculate_control_limits(mean, std, lambda_val):
    """محاسبه UCL و LCL بر اساس فرمول EWMA در حالت پایدار"""
    factor = 3 * std * np.sqrt(lambda_val / (2 - lambda_val))
    ucl = mean + factor
    lcl = mean - factor
    return ucl, lcl

def run_analysis():
    if not os.path.exists(file_path):
        print(f"❌ خطا: فایل یافت نشد در مسیر: {file_path}")
        return

    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df.set_index('date', inplace=True)
        print("✅ مرحله ۱: فایل بارگذاری شد.")
        print(f"📅 بازه زمانی داده‌ها: {df.index.min()} تا {df.index.max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن اکسل: {e}")
        return

    # ۱. جداسازی بازه سلامت و بازه یک ماه اخیر
    try:
        last_date = df.index.max()
        split_date = last_date - pd.Timedelta(days=30)
        baseline_start = split_date - pd.Timedelta(days=30)

        df_baseline = df.loc[baseline_start:split_date].copy()
        df_fault = df.loc[split_date:last_date].copy()
        
        print(f"📊 بازه سلامت: {baseline_start.date()} تا {split_date.date()}")
        print(f"⚠️ بازه خطا: {split_date.date()} تا {last_date.date()}")
        
    except Exception as e:
        print(f"❌ خطا در پردازش تاریخ‌ها: {e}")
        return

    print("⏳ در حال محاسبه شاخص‌ها...")
    print(f"🔧 لاندا (λ) = {LAMBDA}")
    print("="*80)

    target_analysis_results = []

    for col in target_sensors:
        if col not in df_fault.columns:
            print(f"⚠️ سنسور {col} در داده‌ها وجود ندارد.")
            continue
        
        if col not in df_baseline.columns:
            print(f"⚠️ سنسور {col} در داده‌های بیس‌لاین وجود ندارد.")
            continue

        try:
            # محاسبه پارامترهای آماری از بازه سلامت
            mean_base = df_baseline[col].mean()
            std_base = df_baseline[col].std()
            
            # محاسبه UCL و LCL
            ucl, lcl = calculate_control_limits(mean_base, std_base, LAMBDA)
            
            # محاسبه مقادیر EWMA برای بازه خطا
            fault_data = df_fault[col].values
            ewma_values = calculate_ewma(fault_data, LAMBDA)
            current_ewma = ewma_values[-1]
            
            # محاسبه شیب تغییرات
            time_diff_series = df_fault.index.to_series().diff().dt.total_seconds() / 3600.0
            val_diff = df_fault[col].diff()
            instant_slopes = val_diff / time_diff_series
            avg_slope = instant_slopes.mean()
            
            current_raw_value = df_fault[col].iloc[-1]
            
            # ==============================================
            # محاسبه Hours_to_UCL با منطق نهایی:
            # 1. اگر شیب <= 0 -> 0
            # 2. اگر شیب > 0 و مقدار فعلی >= UCL -> 0
            # 3. اگر شیب > 0 و مقدار فعلی < UCL -> (UCL - current) / slope
            # ==============================================
            
            if avg_slope <= 0:
                hours_to_ucl = 0.0
                reason = "شیب منفی یا صفر"
            elif current_raw_value >= ucl:
                hours_to_ucl = 0.0
                reason = "مقدار فعلی از UCL بیشتر یا مساوی است"
            else:
                hours_to_ucl = (ucl - current_raw_value) / avg_slope
                reason = f"محاسبه شد: ({ucl:.4f} - {current_raw_value:.4f}) / {avg_slope:.6f}"
            
            # اطمینان از عدد بودن (NaN یا Inf نباشد)
            if np.isnan(hours_to_ucl) or np.isinf(hours_to_ucl):
                hours_to_ucl = 0.0
                reason = "مقدار نامعتبر (NaN/Inf)"
            else:
                hours_to_ucl = round(hours_to_ucl, 2)
            
            # چاپ دیباگ
            print(f"\n🔍 {col}:")
            print(f"   مقدار فعلی: {current_raw_value:.4f}")
            print(f"   UCL: {ucl:.4f}")
            print(f"   شیب متوسط: {avg_slope:.8f}")
            print(f"   Hours_to_UCL: {hours_to_ucl} ساعت ← {reason}")
            
            out_of_control = "No"
            if current_ewma > ucl or current_ewma < lcl:
                out_of_control = "Yes ⚠️"
            
            target_analysis_results.append({
                'Sensor': col,
                'Current_Raw_Value': round(current_raw_value, 4),
                'Current_EWMA': round(current_ewma, 4),
                'UCL': round(ucl, 4),
                'LCL': round(lcl, 4),
                'Baseline_Mean': round(mean_base, 4),
                'Baseline_Std': round(std_base, 4),
                'Average_Slope_per_Hour': round(avg_slope, 6),
                'Hours_to_UCL': hours_to_ucl,
                'Out_Of_Control': out_of_control
            })
            
        except Exception as e:
            print(f"❌ خطا در پردازش سنسور {col}: {e}")
            continue

    # ۳. رتبه‌بندی RCA
    print("\n" + "="*80)
    print("⏳ در حال محاسبه رتبه‌بندی RCA...")
    
    numeric_cols = df_baseline.select_dtypes(include=[np.number]).columns.tolist()
    rca_list = []
    for c in numeric_cols:
        if c not in df_fault.columns:
            continue
        try:
            mean_base = df_baseline[c].mean()
            std_base = df_baseline[c].std()
            mean_fault = df_fault[c].mean()
            
            if std_base > 0:
                deviation_score = abs(mean_fault - mean_base) / std_base
            else:
                deviation_score = 0
                
            rca_list.append({
                'Sensor': c, 
                'Baseline_Mean': round(mean_base, 4),
                'Fault_Mean': round(mean_fault, 4),
                'Deviation_Score': round(deviation_score, 4)
            })
        except Exception as e:
            continue
    
    rca_summary = pd.DataFrame(rca_list).sort_values('Deviation_Score', ascending=False)
    
    # ۴. ذخیره در اکسل
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)

        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            pd.DataFrame(target_analysis_results).to_excel(writer, sheet_name='Speed_Analysis', index=False)
            rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)
            
            # اضافه کردن توضیحات روش محاسبه
            methodology_note = pd.DataFrame({
                'Parameter': [
                    'Lambda (λ)', 
                    'UCL Formula', 
                    'LCL Formula', 
                    'EWMA Formula', 
                    'Hours_to_UCL Rule'
                ],
                'Value': [
                    f'{LAMBDA}', 
                    f'μ + 3σ√(λ/(2-λ))', 
                    f'μ - 3σ√(λ/(2-λ))', 
                    'E_t = λ·X_t + (1-λ)·E_{t-1}',
                    'If slope>0 and value<UCL → (UCL-value)/slope | Else → 0'
                ]
            })
            methodology_note.to_excel(writer, sheet_name='Methodology', index=False)

        print(f"\n🚀 گزارش با موفقیت ساخته شد:\n{output_filename}")
        
        # آمار نهایی
        positive_slopes = [r for r in target_analysis_results if r['Average_Slope_per_Hour'] > 0]
        positive_hours = [r for r in target_analysis_results if r['Hours_to_UCL'] > 0]
        
        print(f"\n📊 آمار نهایی Hours_to_UCL:")
        print(f"   └─ کل سنسورهای هدف: {len(target_analysis_results)}")
        print(f"   └─ سنسورهای با شیب مثبت: {len(positive_slopes)}")
        print(f"   └─ سنسورهای با زمان مثبت (در حال رسیدن): {len(positive_hours)}")
        print(f"   └─ سنسورهای با زمان صفر: {len(target_analysis_results) - len(positive_hours)}")
        
        if positive_hours:
            print(f"\n📈 سنسورهایی که در حال رسیدن به UCL هستند:")
            for res in positive_hours:
                print(f"   └─ {res['Sensor']}: {res['Hours_to_UCL']} ساعت دیگر (شیب={res['Average_Slope_per_Hour']:.6f})")
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")

if __name__ == "__main__":
    run_analysis()

✅ مرحله ۱: فایل بارگذاری شد.
📅 بازه زمانی داده‌ها: 2021-03-16 05:33:48 تا 2026-05-31 20:30:17
📊 بازه سلامت: 2026-04-01 تا 2026-05-01
⚠️ بازه خطا: 2026-05-01 تا 2026-05-31
⏳ در حال محاسبه شاخص‌ها...
🔧 لاندا (λ) = 0.2

🔍 AssetID_9362:
   مقدار فعلی: 63.0000
   UCL: 62.7222
   شیب متوسط: 0.00747590
   Hours_to_UCL: 0.0 ساعت ← مقدار فعلی از UCL بیشتر یا مساوی است

🔍 AssetID_9363:
   مقدار فعلی: 22.0000
   UCL: 26.4532
   شیب متوسط: 0.00503988
   Hours_to_UCL: 883.58 ساعت ← محاسبه شد: (26.4532 - 22.0000) / 0.005040

🔍 AssetID_9364:
   مقدار فعلی: 70.0000
   UCL: 66.9833
   شیب متوسط: 0.00878710
   Hours_to_UCL: 0.0 ساعت ← مقدار فعلی از UCL بیشتر یا مساوی است

🔍 AssetID_9365:
   مقدار فعلی: 41.0000
   UCL: 60.9987
   شیب متوسط: -0.01803599
   Hours_to_UCL: 0.0 ساعت ← شیب منفی یا صفر

🔍 AssetID_9366:
   مقدار فعلی: 62.0000
   UCL: 69.7112
   شیب متوسط: 0.00965578
   Hours_to_UCL: 798.61 ساعت ← محاسبه شد: (69.7112 - 62.0000) / 0.009656

🔍 AssetID_9367:
   مقدار فعلی: 32.0000
   UCL: 42.8978
  